# 02. Baseline Model Comparison
**Project**: Recut Research Lab 01  
**Benchmark**: CyberAgent AdTEC `ad-consistency`  

### Objective
Establish empirical reference baselines on the AdTEC benchmark:
1. **Majority-Class Baseline**: Predicts most frequent training class.
2. **TF-IDF Char N-Gram + Logistic Regression**: Evaluates surface character representation limits.

### Protocol Reminder
- Training data is used exclusively for fitting.
- Evaluation and error exploration occur strictly on the validation set.
- The official test split remains frozen.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from recut_lab.baseline import MajorityBaseline, TfidfLogRegBaseline
from recut_lab.data import create_synthetic_dataset, load_adtec_dataset
from recut_lab.evaluation import compute_classification_metrics, measure_inference_latency

data_dir = Path('../data/raw/adtec_consistency')
if (data_dir / 'train.jsonl').exists():
    print(f'[+] Loading dataset from {data_dir.resolve()}')
    dataset = load_adtec_dataset(data_dir)
else:
    print('[!] Using synthetic fixture dataset for demonstration.')
    dataset = create_synthetic_dataset(num_samples_per_split=100)

In [ ]:
# Train and evaluate Majority Baseline
majority_model = MajorityBaseline()
majority_model.fit(dataset.train)

maj_preds, maj_latency, maj_time = measure_inference_latency(
    majority_model.predict, dataset.valid, num_runs=3
)
maj_report = compute_classification_metrics(
    y_true=dataset.valid['label'].tolist(),
    y_pred=maj_preds,
    latency_ms_per_sample=maj_latency,
    total_latency_seconds=maj_time,
)

In [ ]:
# Train and evaluate TF-IDF + Logistic Regression Baseline
tfidf_model = TfidfLogRegBaseline(
    ngram_range=(1, 3),
    min_df=2,
    max_features=25000,
    class_weight='balanced',
    random_state=42,
)
tfidf_model.fit(dataset.train)

tfidf_preds, tfidf_latency, tfidf_time = measure_inference_latency(
    tfidf_model.predict, dataset.valid, num_runs=3
)
tfidf_report = compute_classification_metrics(
    y_true=dataset.valid['label'].tolist(),
    y_pred=tfidf_preds,
    latency_ms_per_sample=tfidf_latency,
    total_latency_seconds=tfidf_time,
)

In [ ]:
# Comparative Metrics Table
comparison_df = pd.DataFrame([
    {
        'Model': 'Majority Class',
        'Accuracy': maj_report.accuracy,
        'Macro-F1': maj_report.macro_f1,
        'Consistent F1': maj_report.per_class_f1.get('consistent', 0.0),
        'Inconsistent F1': maj_report.per_class_f1.get('inconsistent', 0.0),
        'Latency (ms/sample)': maj_report.inference_latency_ms_per_sample,
    },
    {
        'Model': 'TF-IDF (1-3 chars) + LogReg',
        'Accuracy': tfidf_report.accuracy,
        'Macro-F1': tfidf_report.macro_f1,
        'Consistent F1': tfidf_report.per_class_f1.get('consistent', 0.0),
        'Inconsistent F1': tfidf_report.per_class_f1.get('inconsistent', 0.0),
        'Latency (ms/sample)': tfidf_report.inference_latency_ms_per_sample,
    }
])
print(comparison_df.to_string(index=False))

### Scientific Interpretation
- **Macro-F1 Baseline**: Majority voting reveals the nominal threshold under class imbalance.
- **Surface Feature Capacity**: TF-IDF character n-grams test whether lexical matching is sufficient or whether deep contextual modeling (e.g. cross-encoders) is necessary for high consistency accuracy.